In [61]:
import sklearn
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV
from sklearn.model_selection import RepeatedKFold


In [77]:
scaler = StandardScaler()

test = pd.read_csv("wine_test.csv")
test_scaled = scaler.fit_transform(test)
test_scaled = sm.add_constant(test_scaled)

In [95]:
#FIRST ORDER MODEL
#load data
df = pd.read_csv("wine_training.csv")

#TASK ONE
scaled_data = scaler.fit_transform(df)

#TASK TWO
X = scaled_data[:, 0:11]
print(X.shape)
y = scaled_data[:, 11]
print(y.shape)

X = sm.add_constant(X)
print(X.shape)
model = sm.OLS(y, X)
results = model.fit()

y_pred_train = results.predict(X)
print("MSE:", mean_squared_error(y, y_pred_train))

#TASK THREE
largest = np.argsort(np.abs(results.params))[::-1]
print("Most important features:")
for i in largest[:5]:
    print(f"Feature {i - 1}: {results.params[i]}")

#Test data
y_test = test_scaled[:, 12]
x_test = test_scaled[:, 0:12]
y_test_pred = results.predict(x_test)

print("test MSE:", mean_squared_error(test_scaled[:, 12], y_test_pred))

(1114, 11)
(1114,)
(1114, 12)
MSE: 0.6276759453927347
Most important features:
Feature 10: 0.3490433593419763
Feature 1: -0.3034357431827308
Feature 9: 0.21272802336198232
Feature 6: -0.19258580436940198
Feature 4: -0.1446012586456653
test MSE: 0.7018782862693438


In [ ]:
#SECOND ORDER MODEL
#Task Four
for i in range(11):
    X = np.hstack((X, X[:, i + 1].reshape(-1, 1) ** 2))

model = sm.OLS(y, X)
results = model.fit()

#calculate predictions and MSE
y_pred_train = results.predict(X)
print("MSE:", mean_squared_error(y, y_pred_train))

#process test data
for i in range(11):
    x_test = np.hstack((x_test, x_test[:, i + 1].reshape(-1, 1) ** 2))  
y_test = test_scaled[:, 12]
y_test_pred = results.predict(x_test)

#calculate test MSE
print("test MSE:", mean_squared_error(test_scaled[:, 12], y_test_pred))

MSE: 0.596045904457033
test MSE: 0.7152677586453738


In [ ]:
#INTERACTION MODEL
#Task Six
#load data
df = pd.read_csv("wine_training.csv")

#scale data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

#split into X and Y
full_X = scaled_data[:, 0:11]
full_y = scaled_data[:, 11]

X = full_X
y = full_y

#create columns for interactions between the 5 most important features
largest_coefs = [1, 4, 6, 9, 10]
for coef in largest_coefs:
    largest_coefs = largest_coefs[1:]  # Remove the current coef from the list
    for coef2 in largest_coefs:
        if coef != coef2:
            X = np.hstack((X, (X[:, coef].reshape(-1, 1) * X[:, coef2].reshape(-1, 1))))
    
#create model
X = sm.add_constant(X)
model = sm.OLS(y, X)
results = model.fit()

#calculate MSE
y_pred_train = results.predict(X)
print("MSE:", mean_squared_error(y, y_pred_train))

#process test data
y_test = test_scaled[:, 12]
x_test = test_scaled[:, 0:12]

largest_coefs = [1, 4, 6, 9, 10]
for coef in largest_coefs:
    largest_coefs = largest_coefs[1:]  # Remove the current coef from the list
    for coef2 in largest_coefs:
        if coef != coef2:
            x_test = np.hstack((x_test, (x_test[:, coef].reshape(-1, 1) * x_test[:, coef2].reshape(-1, 1))))

x_test = sm.add_constant(x_test)
y_test_pred = results.predict(x_test)

#calculate test MSE
print("test MSE:", mean_squared_error(test_scaled[:, 12], y_test_pred))

MSE: 0.5957338317044855
test MSE: 0.7227834486477196


In [96]:
#Task Seven
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)

#create lasso model on limited features
model = LassoCV(alphas = [0.01, 1, 10, 100], cv=cv, n_jobs = -1)
model.fit(X, y)
print("Best alpha:", model.alpha_)

#refit model with best alpha on full dataset
model.fit(full_X, full_y)
print(model.coef_)
print("MSE:", mean_squared_error(full_y, model.predict(full_X)))

#calculate test MSE
print("test MSE:", mean_squared_error(test_scaled[:, 12], model.predict(test_scaled[:, 0:11])))

Best alpha: 0.01
[ 0.         -0.28124315 -0.03667249  0.         -0.13313975  0.05641368
 -0.16593253  0.         -0.05163414  0.19880602  0.32493705]
MSE: 0.6299069546027296
test MSE: 1.2038253798383631
